In [20]:
# --- 1. IMPORTS ---

import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import glob
from io import BytesIO
from spark_session import get_spark_session

# Add bucket module to path
sys.path.insert(0, os.path.abspath('../../src/bucket'))
from r2_config import get_r2_client, BUCKET_NAME

print("✓ Imports completed.")
print(f"✓ R2 Bucket: {BUCKET_NAME}")

✓ Imports completed.
✓ R2 Bucket: apache-spark-perception-tree-rings


In [21]:
# --- 2. CONFIGURATION PARAMETERS ---

# Host paths (for Driver to read files)
HOST_DATA_PATH = "./../../data" 
# Container paths (inside Docker as in docker-compose.yml)
CONTAINER_DATA_PATH = "/opt/spark/data"

# R2 bucket prefix for processed images
UPLOAD_PREFIX = 'processed/'

# Slice configuration
FROM_N_SLICES = 3
TO_N_SLICES = 30

# Limit number of images to process (for testing)
IMAGE_LIMIT = 64

# Upload mode: True = upload to R2, False = save locally
UPLOAD_TO_R2 = True

print(f"Configuration:")
print(f"  - Processing slices from {FROM_N_SLICES} to {TO_N_SLICES}")
print(f"  - Image limit: {IMAGE_LIMIT}")
print(f"  - Upload to R2: {UPLOAD_TO_R2}")
print(f"  - R2 prefix: {UPLOAD_PREFIX}")

Configuration:
  - Processing slices from 3 to 30
  - Image limit: 64
  - Upload to R2: True
  - R2 prefix: processed/


In [22]:
# --- 3. INICIALIZAR SESIÓN DE SPARK ---
print("Iniciando SparkSession en modo distribuido...")
spark = get_spark_session("TreeRingSlicing (Distributed)")

sc = spark.sparkContext
print("\n--- SparkSession Iniciada --- ✅")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"UI Web: {sc.uiWebUrl}")

Iniciando SparkSession en modo distribuido...
Attempting to connect to master at: spark://localhost:7077
Will announce driver host IP as: 172.26.0.1
Check: IP 172.26.0.1 resolves locally.

--- Connection Successful! --- ✅
SparkSession object: <pyspark.sql.session.SparkSession object at 0x7fbf11f680d0>
SparkContext object: <SparkContext master=spark://localhost:7077 appName=TreeRingSlicing (Distributed)>
Spark version in use: 4.0.1

--- SparkSession Iniciada --- ✅
Spark Version: 4.0.1
Master: spark://localhost:7077
UI Web: http://172.26.0.1:4040


In [23]:
# --- 4. HELPER FUNCTIONS ---

def load_pith_locations(metadata_path):
    """
    Load pith center locations from CSV into a dictionary for fast lookup.
    """
    try:
        csv_path = os.path.join(metadata_path, 'pith_location.csv')
        df = pd.read_csv(csv_path)
        # Create dictionary: {'F10b': (cx, cy)}
        pith_dict = {row['Image']: (row['cx'], row['cy']) for index, row in df.iterrows()}
        
        print(f"  ✓ Loaded {len(pith_dict)} pith locations from CSV")
        return pith_dict
        
    except FileNotFoundError:
        print(f"  ✗ Error: CSV file not found at {csv_path}")
        return {}
    except Exception as e:
        print(f"  ✗ Error loading CSV: {e}")
        return {}


def load_ring_counts(metadata_path):
    """
    Load ring counts from CSV into a dictionary for fast lookup.
    """
    try:
        csv_path = os.path.join(metadata_path, 'ring_counts.csv')
        df = pd.read_csv(csv_path)
        # Create dictionary: {'F10b': 23}
        ring_dict = {row['image_code']: int(row['rings_count']) for index, row in df.iterrows()}
        
        print(f"  ✓ Loaded {len(ring_dict)} ring counts from CSV")
        return ring_dict
        
    except FileNotFoundError:
        print(f"  ✗ Error: CSV file not found at {csv_path}")
        return {}
    except Exception as e:
        print(f"  ✗ Error loading CSV: {e}")
        return {}


def get_image_files(raw_path, limit=None):
    """
    Scan the 'raw' folder and return a list of image filenames.
    """
    try:
        all_files = [f for f in os.listdir(raw_path) if f.lower().endswith('.png')]
        all_files.sort()
        
        if limit is not None:
            print(f"  ✓ Found {len(all_files)} images. Processing first {limit}")
            return all_files[:limit]
        else:
            print(f"  ✓ Found and will process {len(all_files)} images")
            return all_files
            
    except FileNotFoundError:
        print(f"  ✗ Error: 'raw' directory not found at {raw_path}")
        return []

In [24]:
# --- 5. WORKER PROCESSING LOGIC ---

def process_image_multi_slices_r2(image_name, base_data_path, upload_prefix,
                                    from_n_slices, to_n_slices,
                                    pith_broadcast, ring_counts_broadcast,
                                    bucket_name):
    """
    Process a single image and generate slices for all N values (from_n_slices to to_n_slices).
    Each slice is cropped to its bounding box and uploaded directly to R2.
    Returns a list of (image_path, ring_count) tuples for CSV generation.
    
    This function will run in parallel on Spark workers.
    """
    try:
        # Import R2 dependencies inside worker (needed for Spark serialization)
        import boto3
        import os
        from io import BytesIO
        
        # --------------------------------------------------
        # 1. Setup R2 connection
        # --------------------------------------------------
        # Get R2 credentials from environment (workers inherit from master)
        endpoint_url = f"https://{os.getenv('CLOUDFLARE_ACCOUNT_ID')}.r2.cloudflarestorage.com"
        s3_client = boto3.client(
            "s3",
            endpoint_url=endpoint_url,
            aws_access_key_id=os.getenv('CLOUDFLARE_ACCESS_KEY'),
            aws_secret_access_key=os.getenv('CLOUDFLARE_SECRET_KEY'),
            region_name="auto",
        )
        
        # --------------------------------------------------
        # 2. Setup paths and load image
        # --------------------------------------------------
        image_path = os.path.join(base_data_path, 'raw', image_name)
        image_name_no_ext = os.path.splitext(image_name)[0]
        
        # Get broadcast dictionaries
        pith_map = pith_broadcast.value
        ring_counts_map = ring_counts_broadcast.value
        
        # Load image
        img = cv2.imread(image_path)
        if img is None:
            return (image_name, "ERROR: Could not read image", [])
            
        h, w, _ = img.shape
        
        # --------------------------------------------------
        # 3. Get metadata for this image
        # --------------------------------------------------
        if image_name_no_ext not in pith_map:
            return (image_name, f"ERROR: Pith center not found for {image_name_no_ext}", [])
        
        if image_name_no_ext not in ring_counts_map:
            return (image_name, f"ERROR: Ring count not found for {image_name_no_ext}", [])
            
        # Get pith center (cy, cx format from CSV)
        (cy, cx) = pith_map[image_name_no_ext]
        
        # Get ring count (all slices inherit this value)
        ring_count = ring_counts_map[image_name_no_ext]
        
        # Calculate maximum radius to cover entire image from center
        radius = int(np.sqrt(max(cx, w - cx)**2 + max(cy, h - cy)**2)) + 1
        
        # --------------------------------------------------
        # 4. Generate and upload slices for each N value
        # --------------------------------------------------
        total_slices_uploaded = 0
        total_original_pixels = 0
        total_cropped_pixels = 0
        total_bytes_uploaded = 0
        
        # List to store (image_path, ring_count) for CSV
        uploaded_images = []
        
        for n_slices in range(from_n_slices, to_n_slices + 1):
            angle_per_slice = 360.0 / n_slices
            
            for i in range(n_slices):
                # Calculate angles
                start_angle = (i * angle_per_slice) - 90
                end_angle = ((i + 1) * angle_per_slice) - 90
                
                # Create mask
                mask = np.zeros((h, w), dtype=np.uint8)
                cv2.ellipse(
                    mask,
                    center=(int(cx), int(cy)),
                    axes=(int(radius), int(radius)),
                    angle=0,
                    startAngle=start_angle,
                    endAngle=end_angle,
                    color=255,
                    thickness=-1
                )
                
                # Apply mask
                result = cv2.bitwise_and(img, img, mask=mask)
                
                # Crop to bounding box
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                if contours:
                    x, y, bbox_w, bbox_h = cv2.boundingRect(contours[0])
                    cropped_result = result[y:y+bbox_h, x:x+bbox_w]
                    
                    # Track pixel savings
                    total_original_pixels += (w * h)
                    total_cropped_pixels += (bbox_w * bbox_h)
                else:
                    cropped_result = result
                
                # --------------------------------------------------
                # 5. Encode image to PNG in memory and upload to R2
                # --------------------------------------------------
                # Encode to PNG format
                success, encoded_image = cv2.imencode('.png', cropped_result)
                if not success:
                    return (image_name, f"ERROR: Failed to encode slice {i+1}", [])
                
                # Convert to bytes
                image_bytes = encoded_image.tobytes()
                total_bytes_uploaded += len(image_bytes)
                
                # Construct S3 key (path in bucket)
                # Format: processed/[N]/[image_name]/[image_name]_[slice_number].png
                s3_key = f"{upload_prefix}{n_slices}/{image_name_no_ext}/{image_name_no_ext}_{i+1}.png"
                
                # Upload to R2
                s3_client.put_object(
                    Bucket=bucket_name,
                    Key=s3_key,
                    Body=image_bytes,
                    ContentType='image/png'
                )
                
                # Store path and ring_count for CSV
                uploaded_images.append((s3_key, ring_count))
                
                total_slices_uploaded += 1
        
        # Calculate statistics
        space_savings = 0
        if total_original_pixels > 0:
            space_savings = ((total_original_pixels - total_cropped_pixels) / total_original_pixels) * 100
        
        total_mb = total_bytes_uploaded / (1024 * 1024)
        n_range = to_n_slices - from_n_slices + 1
        
        status_msg = (f"SUCCESS: {total_slices_uploaded} slices uploaded to R2 "
                     f"({n_range} N values) "
                     f"| Ring count: {ring_count} "
                     f"| Total size: {total_mb:.2f} MB "
                     f"| Space saved: {space_savings:.1f}%")
        
        return (image_name, status_msg, uploaded_images)
        
    except Exception as e:
        import traceback
        return (image_name, f"ERROR: {str(e)}\n{traceback.format_exc()}", [])

In [25]:
# --- 6. SPARK PIPELINE EXECUTION ---

print("=" * 80)
print("MULTI-SLICE IMAGE PROCESSING PIPELINE - R2 UPLOAD")
print("=" * 80)

# ------------------------------------------------------------------------------
# STEP 0: Test R2 connection (Driver-side)
# ------------------------------------------------------------------------------
print("\n[0/6] Testing R2 connection...")

try:
    r2_client = get_r2_client()
    # Test connection by listing bucket
    r2_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"  ✓ Successfully connected to R2 bucket: {BUCKET_NAME}")
except Exception as e:
    raise Exception(f"Failed to connect to R2: {e}")

# ------------------------------------------------------------------------------
# STEP 1: Load and broadcast metadata (Driver-side)
# ------------------------------------------------------------------------------
print("\n[1/6] Loading metadata (Driver-side)...")

# Load pith locations
pith_dict = load_pith_locations(os.path.join(HOST_DATA_PATH, 'metadata'))
if not pith_dict:
    raise ValueError("Failed to load pith data. Aborting.")

# Load ring counts
ring_counts_dict = load_ring_counts(os.path.join(HOST_DATA_PATH, 'metadata'))
if not ring_counts_dict:
    raise ValueError("Failed to load ring counts data. Aborting.")

# Broadcast to workers
pith_broadcast = sc.broadcast(pith_dict)
ring_counts_broadcast = sc.broadcast(ring_counts_dict)

print(f"  ✓ Broadcast {len(pith_dict)} pith locations")
print(f"  ✓ Broadcast {len(ring_counts_dict)} ring counts")
print(f"  Example: F02a has {ring_counts_dict.get('F02a', 'N/A')} rings")

# ------------------------------------------------------------------------------
# STEP 2: Get list of images to process (Driver-side)
# ------------------------------------------------------------------------------
print("\n[2/6] Scanning for image files (Driver-side)...")

image_files = get_image_files(os.path.join(HOST_DATA_PATH, 'raw'), limit=IMAGE_LIMIT)
if not image_files:
    raise ValueError("No images found to process. Aborting.")

print(f"  ✓ Found {len(image_files)} image(s) to process")

# ------------------------------------------------------------------------------
# STEP 3: Calculate expected output
# ------------------------------------------------------------------------------
print("\n[3/6] Calculating expected output...")

total_n_values = TO_N_SLICES - FROM_N_SLICES + 1
total_slices_per_image = sum(range(FROM_N_SLICES, TO_N_SLICES + 1))
total_files_expected = len(image_files) * total_slices_per_image

print(f"  → N values to process: {total_n_values} (from {FROM_N_SLICES} to {TO_N_SLICES})")
print(f"  → Slices per image: {total_slices_per_image}")
print(f"  → Total files to upload: {total_files_expected}")

# Memory estimation
avg_label_size = 100  # bytes per label (image_path + ring_count)
estimated_labels_mb = (total_files_expected * avg_label_size) / (1024 * 1024)
print(f"  → Estimated labels size in memory: {estimated_labels_mb:.2f} MB")

if estimated_labels_mb > 100:
    print(f"  ⚠️  Large dataset detected! Using optimized collection strategy")

# ------------------------------------------------------------------------------
# STEP 4: Create RDD and define worker function
# ------------------------------------------------------------------------------
print("\n[4/6] Creating RDD for parallel processing...")

# Increase partitions for better parallelism with large datasets
num_partitions = max(len(image_files), 8)  # At least 8 partitions
image_rdd = sc.parallelize(image_files, numSlices=num_partitions)
print(f"  ✓ RDD created with {image_rdd.getNumPartitions()} partition(s)")
print(f"  ℹ️  Each worker will process ~{len(image_files) // num_partitions} images")


def run_multi_slicing_r2(image_name):
    """
    Wrapper function that will run on each Spark worker.
    Processes image and uploads directly to R2.
    """
    return process_image_multi_slices_r2(
        image_name,
        CONTAINER_DATA_PATH,
        UPLOAD_PREFIX,
        FROM_N_SLICES,
        TO_N_SLICES,
        pith_broadcast,
        ring_counts_broadcast,
        BUCKET_NAME
    )

# ------------------------------------------------------------------------------
# STEP 5: Execute processing and upload (with progress tracking)
# ------------------------------------------------------------------------------
print("\n[5/6] Executing distributed multi-slice generation and R2 upload...")
print(f"  ⏳ Processing {len(image_files)} image(s) × {total_n_values} N values...")
print(f"  🌐 Uploading to: s3://{BUCKET_NAME}/{UPLOAD_PREFIX}")
print(f"  💡 Processing in batches to optimize memory usage...")

# Process and collect results
# Note: collect() brings all results to driver, but they're just status messages + labels
# Labels are small (just paths + numbers), so this should be fine
results = image_rdd.map(run_multi_slicing_r2).collect()

print(f"  ✓ Processing and upload completed!")

# ------------------------------------------------------------------------------
# STEP 6: Display execution results and collect labels
# ------------------------------------------------------------------------------
print("\n[6/6] Execution Summary:")
print("=" * 80)

success_count = 0
error_count = 0
total_mb_uploaded = 0

# Collect all (image_path, ring_count) tuples for labels CSV
# This is memory-efficient as each tuple is just ~100 bytes
all_labels = []

for image_name, status, uploaded_images in results:
    status_icon = "✓" if "SUCCESS" in status else "✗"
    # Only print first 10 and last 10 to avoid flooding console
    if success_count + error_count < 10 or success_count + error_count >= len(results) - 10:
        print(f"  {status_icon} {image_name}")
        print(f"     {status}")
    elif success_count + error_count == 10:
        print(f"  ... processing {len(results) - 20} more images ...")
    
    if "SUCCESS" in status:
        success_count += 1
        # Collect labels from this image
        all_labels.extend(uploaded_images)
        
        # Extract MB from status message
        if "MB" in status:
            try:
                mb_str = status.split("Total size: ")[1].split(" MB")[0]
                total_mb_uploaded += float(mb_str)
            except:
                pass
    else:
        error_count += 1

print("=" * 80)
print(f"\n📊 Final Statistics:")
print(f"  ✓ Successful: {success_count}")
print(f"  ✗ Errors: {error_count}")
print(f"  📁 Total files uploaded: {len(all_labels)}")
print(f"  💾 Total data uploaded: {total_mb_uploaded:.2f} MB")
print(f"  📊 Actual labels collected: {len(all_labels)} entries")
print(f"  🌐 R2 location: s3://{BUCKET_NAME}/{UPLOAD_PREFIX}")

if len(all_labels) != total_files_expected:
    print(f"  ⚠️  Warning: Expected {total_files_expected} files but got {len(all_labels)}")

print("\n✅ Pipeline execution completed!")
print("=" * 80)

MULTI-SLICE IMAGE PROCESSING PIPELINE - R2 UPLOAD

[0/6] Testing R2 connection...
  ✓ Successfully connected to R2 bucket: apache-spark-perception-tree-rings

[1/6] Loading metadata (Driver-side)...
  ✓ Loaded 64 pith locations from CSV
  ✓ Loaded 64 ring counts from CSV
  ✓ Broadcast 64 pith locations
  ✓ Broadcast 64 ring counts
  Example: F02a has 23 rings

[2/6] Scanning for image files (Driver-side)...
  ✓ Found 64 images. Processing first 64
  ✓ Found 64 image(s) to process

[3/6] Calculating expected output...
  → N values to process: 28 (from 3 to 30)
  → Slices per image: 462
  → Total files to upload: 29568
  → Estimated labels size in memory: 2.82 MB

[4/6] Creating RDD for parallel processing...
  ✓ RDD created with 64 partition(s)
  ℹ️  Each worker will process ~1 images

[5/6] Executing distributed multi-slice generation and R2 upload...
  ⏳ Processing 64 image(s) × 28 N values...
  🌐 Uploading to: s3://apache-spark-perception-tree-rings/processed/
  💡 Processing in batch

  ✓ Processing and upload completed!

[6/6] Execution Summary:
  ✓ F02a.png
     SUCCESS: 462 slices uploaded to R2 (28 N values) | Ring count: 23 | Total size: 244.67 MB | Space saved: 83.1%
  ✓ F02b.png
     SUCCESS: 462 slices uploaded to R2 (28 N values) | Ring count: 22 | Total size: 117.23 MB | Space saved: 83.1%
  ✓ F02c.png
     SUCCESS: 462 slices uploaded to R2 (28 N values) | Ring count: 22 | Total size: 302.77 MB | Space saved: 83.1%
  ✓ F02d.png
     SUCCESS: 462 slices uploaded to R2 (28 N values) | Ring count: 20 | Total size: 249.66 MB | Space saved: 83.1%
  ✓ F02e.png
     SUCCESS: 462 slices uploaded to R2 (28 N values) | Ring count: 20 | Total size: 175.47 MB | Space saved: 83.1%
  ✓ F03a.png
     SUCCESS: 462 slices uploaded to R2 (28 N values) | Ring count: 24 | Total size: 265.83 MB | Space saved: 83.1%
  ✓ F03b.png
     SUCCESS: 462 slices uploaded to R2 (28 N values) | Ring count: 23 | Total size: 131.58 MB | Space saved: 83.1%
  ✓ F03c.png
     SUCCESS: 462 sli

In [26]:
# --- 7. GENERATE AND UPLOAD LABELS CSV ---

print("\n" + "=" * 80)
print("GENERATING LABELS CSV FOR CNN TRAINING")
print("=" * 80)

try:
    # Create DataFrame from collected labels
    labels_df = pd.DataFrame(all_labels, columns=['image_path', 'rings_count'])
    
    print(f"\n📊 Labels DataFrame:")
    print(f"  → Total rows: {len(labels_df)}")
    print(f"  → Unique ring counts: {labels_df['rings_count'].nunique()}")
    print(f"  → Ring count distribution:")
    print(labels_df['rings_count'].value_counts().sort_index())
    
    # Show sample rows
    print(f"\n📋 Sample rows (first 10):")
    print(labels_df.head(10).to_string(index=False))
    
    # Save to CSV in memory
    from io import StringIO
    csv_buffer = StringIO()
    labels_df.to_csv(csv_buffer, index=False)
    csv_content = csv_buffer.getvalue().encode('utf-8')
    
    # Upload labels CSV to R2
    labels_s3_key = 'metadata/slice_labels.csv'
    r2_client.put_object(
        Bucket=BUCKET_NAME,
        Key=labels_s3_key,
        Body=csv_content,
        ContentType='text/csv',
        Metadata={
            'description': 'Labels for tree ring slice images - CNN training dataset',
            'total_images': str(len(labels_df)),
            'n_slices_range': f'{FROM_N_SLICES}-{TO_N_SLICES}',
            'uploaded_by': 'spark-pipeline'
        }
    )
    
    csv_size_kb = len(csv_content) / 1024
    print(f"\n✓ Uploaded labels CSV to s3://{BUCKET_NAME}/{labels_s3_key}")
    print(f"  Size: {csv_size_kb:.2f} KB")
    print(f"  Records: {len(labels_df)}")
    
    # Also upload the original ring_counts CSV for reference
    print(f"\n📤 Uploading original ring_counts metadata...")
    csv_path = os.path.join(HOST_DATA_PATH, 'metadata', 'ring_counts.csv')
    
    with open(csv_path, 'rb') as csv_file:
        csv_content = csv_file.read()
    
    s3_key = 'metadata/ring_counts_original.csv'
    r2_client.put_object(
        Bucket=BUCKET_NAME,
        Key=s3_key,
        Body=csv_content,
        ContentType='text/csv',
        Metadata={
            'description': 'Original ring counts by image - source metadata',
            'uploaded_by': 'spark-pipeline'
        }
    )
    
    csv_size_kb = len(csv_content) / 1024
    print(f"✓ Uploaded ring_counts_original.csv to s3://{BUCKET_NAME}/metadata/")
    print(f"  Size: {csv_size_kb:.2f} KB")
    print(f"  Records: {len(ring_counts_dict)}")
    
except Exception as e:
    print(f"✗ Error generating/uploading CSV: {e}")
    import traceback
    traceback.print_exc()

print("=" * 80)


GENERATING LABELS CSV FOR CNN TRAINING

📊 Labels DataFrame:
  → Total rows: 29568
  → Unique ring counts: 10
  → Ring count distribution:
rings_count
13     462
14    2310
15    2772
16    6006
17    2772
20    1386
21    2310
22    4158
23    4158
24    3234
Name: count, dtype: int64

📋 Sample rows (first 10):
                 image_path  rings_count
processed/3/F02a/F02a_1.png           23
processed/3/F02a/F02a_2.png           23
processed/3/F02a/F02a_3.png           23
processed/4/F02a/F02a_1.png           23
processed/4/F02a/F02a_2.png           23
processed/4/F02a/F02a_3.png           23
processed/4/F02a/F02a_4.png           23
processed/5/F02a/F02a_1.png           23
processed/5/F02a/F02a_2.png           23
processed/5/F02a/F02a_3.png           23

✓ Uploaded labels CSV to s3://apache-spark-perception-tree-rings/metadata/slice_labels.csv
  Size: 935.83 KB
  Records: 29568

📤 Uploading original ring_counts metadata...
✓ Uploaded ring_counts_original.csv to s3://apache-spark-perce

In [27]:
# --- 8. VERIFICATION: List uploaded files in R2 ---

print("\n" + "=" * 80)
print("VERIFICATION: Checking R2 bucket structure")
print("=" * 80)

try:
    # List objects by N value prefix
    print(f"\nListing objects in s3://{BUCKET_NAME}/{UPLOAD_PREFIX}...\n")
    
    for n_slices in range(FROM_N_SLICES, TO_N_SLICES + 1):
        prefix = f"{UPLOAD_PREFIX}{n_slices}/"
        
        # List objects with this prefix
        response = r2_client.list_objects_v2(
            Bucket=BUCKET_NAME,
            Prefix=prefix,
            MaxKeys=1000
        )
        
        if 'Contents' in response:
            total_size_mb = sum(obj['Size'] for obj in response['Contents']) / (1024 * 1024)
            print(f"✓ N={n_slices:2d}: {len(response['Contents'])} file(s), {total_size_mb:.2f} MB")
            
            # Show first few files as examples
            if len(response['Contents']) > 0:
                print(f"     Examples:")
                for obj in response['Contents'][:3]:
                    key = obj['Key']
                    size_kb = obj['Size'] / 1024
                    print(f"       - {key.split('/')[-1]} ({size_kb:.1f} KB)")
        else:
            print(f"✗ N={n_slices:2d}: No files found")
    
    # Check metadata CSV
    print(f"\n✓ Checking metadata CSV...")
    try:
        response = r2_client.head_object(Bucket=BUCKET_NAME, Key='metadata/ring_counts.csv')
        size_kb = response['ContentLength'] / 1024
        print(f"  ✓ Found: metadata/ring_counts.csv ({size_kb:.2f} KB)")
    except:
        print(f"  ✗ metadata/ring_counts.csv not found")
    
except Exception as e:
    print(f"✗ Error listing R2 objects: {e}")

print("\n" + "=" * 80)


VERIFICATION: Checking R2 bucket structure

Listing objects in s3://apache-spark-perception-tree-rings/processed/...

✓ N= 3: 192 file(s), 392.66 MB
     Examples:
       - F02a_1.png (2528.4 KB)
       - F02a_2.png (2973.2 KB)
       - F02a_3.png (3193.7 KB)
✓ N= 4: 256 file(s), 391.13 MB
     Examples:
       - F02a_1.png (1980.9 KB)
       - F02a_2.png (1950.8 KB)
       - F02a_3.png (2323.4 KB)
✓ N= 5: 320 file(s), 394.46 MB
     Examples:
       - F02a_1.png (1678.0 KB)
       - F02a_2.png (1513.0 KB)
       - F02a_3.png (1540.3 KB)
✓ N= 6: 384 file(s), 394.59 MB
     Examples:
       - F02a_1.png (1452.1 KB)
       - F02a_2.png (1091.6 KB)
       - F02a_3.png (1419.4 KB)
✓ N= 7: 448 file(s), 396.33 MB
     Examples:
       - F02a_1.png (1251.6 KB)
       - F02a_2.png (969.9 KB)
       - F02a_3.png (1237.3 KB)
✓ N= 8: 512 file(s), 396.95 MB
     Examples:
       - F02a_1.png (1090.7 KB)
       - F02a_2.png (916.6 KB)
       - F02a_3.png (926.6 KB)
✓ N= 9: 576 file(s), 398.24 MB
 

In [28]:
# --- 9. STOP SPARK SESSION ---

print("\n🛑 Stopping SparkSession...")
spark.stop()
print("✓ Session stopped.")
print("\n" + "=" * 80)
print("🎉 ALL TASKS COMPLETED SUCCESSFULLY!")
print("=" * 80)


🛑 Stopping SparkSession...
✓ Session stopped.

🎉 ALL TASKS COMPLETED SUCCESSFULLY!
